# Track 08b (심화) — 나의 골든셋 만들기

### 골든셋(golden set)이란?

골든셋은 **팀이 합의한 고정 테스트셋**입니다. 정답, 필수 키, 근거 컨텍스트, 기대 도구가 붙어 있는 사례 묶음이며, 릴리스할 때마다 **같은 입력**으로 지표를 측정합니다. 점수가 기준값 아래로 떨어지면 배포를 멈추는 **회귀 게이트**의 기준이 됩니다. 08a가 M1–M10이 *무엇을 측정하는지*를 다뤘다면, 08b는 그 지표를 사용해 **우리 팀의 골든셋을 만들고 회귀 게이트를 적용하는 방법**을 보여줍니다.

### 이 노트북에서 보여줄 것

`golden_seed.jsonl`(22건)을 카테고리별 합성 trial로 바꿔 **오프라인**(키 없이) 채점합니다. 그런 다음 mini-check로 회귀 게이트 기준을 정하고, **게이트가 회귀를 감지해 배포를 막는 장면**까지 확인합니다. API 키가 있으면 실제 데이터셋과 τ-bench로도 연결할 수 있습니다.

| Session | 무엇을 | 보여주는 내용 / 이유 |
|---|---|---|
| **1. 채점** | 시드 → trial → M1/M5/M6/M9/M3 | 카테고리별로 5개 지표를 계산해 **모든 행이 평가에 쓰이도록** 합니다. |
| **2. 게이트** | mini-check | 모든 행 채점 + M1·M6·M5≥0.8 + M9 건수 기준으로 회귀 게이트를 정합니다. |
| **2b. 게이트 시연** | 회귀된 후보 → FAIL | 지표가 떨어졌을 때 게이트가 **배포를 막는** 과정을 직접 확인합니다. |
| **3. (선택) 데이터셋** | BFCL·IFEval·τ-bench 미리보기 | 표준 벤치마크를 로드하고, 캐시가 없으면 오류 없이 건너뜁니다. |
| **4. (선택) τ-bench E2E** | eval.run pass^k | **기본 비활성화**(즉시 종료). 플래그를 켜면 실제 멀티턴 시뮬레이션을 실행합니다(수 분 소요). |
| **5. 산출물** | my_golden.jsonl + regression_report.json | 재사용할 골든셋과 이번 실행의 회귀 리포트를 분리해 저장합니다. |

### 이 노트북을 마치면

- 정답, 필수 키, 근거, 기대 도구가 붙은 **팀 골든셋**을 만들고, M1/M5/M6/M9/M3로 회귀를 채점할 수 있습니다.
- `cases≥20` 같은 단순 건수 대신 **모든 행이 실제로 채점되는지**를 기준으로 게이트를 거는 방법을 익힙니다.
- 골든셋(라벨)과 회귀 리포트(점수)를 분리해 산출물로 남기고, 필요하면 실제 데이터셋으로 확장할 수 있습니다.

> **한 줄 요약:** 지표를 *우리 팀 골든셋*에 적용해, 점수가 떨어지면 배포를 멈추는 **릴리스 회귀 게이트**로 만듭니다.

- **구성:** 각 `Session`은 가이드(텍스트) → 코드 → 출력 해석(텍스트) 순입니다.
- **필요:** 오프라인 회귀(Session 1–2b·5)는 **키 없이** 실행됩니다(로컬 `eval/` 패키지 필요). Session 3 미리보기는 벤치마크 캐시가 있으면 더 풍부하게 보이고, Session 4 τ-bench E2E는 멀티턴이라 느리므로 **기본 비활성화** 상태입니다. 실제로 실행하려면 `RUN_TAUBENCH_E2E=True`, `EXAONE_API_KEY`, tau-bench 별도 설치가 모두 필요합니다.
- **산출물:** `recipes/track08_evaluation_m1_m10/_out/my_golden.jsonl`(골든셋), `recipes/track08_evaluation_m1_m10/_out/regression_report.json`(회귀 점수)

In [ ]:
import json
import os
import sys
from datetime import datetime, timezone

import logging
import warnings

# (en) Quiet library logs for readable notebook output.
# (kr) 노트북 출력이 읽기 쉽도록 라이브러리 로그를 줄입니다.
for _log_name in ("exaone", "exaone.llm", "exaone.llm.exaone_client", "urllib3"):
    logging.getLogger(_log_name).setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="Unverified HTTPS request")

# (en) Requires editable install at repo root: pip install -r requirements.txt && pip install -e ./exaone
# (kr) 저장소 루트에서 editable 설치가 필요합니다: pip install -r requirements.txt && pip install -e ./exaone
try:
    import exaone
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "exaone이 설치되지 않았습니다. 저장소 루트에서 "
        "pip install -r requirements.txt && pip install -e ./exaone 후 커널을 재시작하세요."
    ) from exc

exaone.load_project_env()
# (en) Gate the optional live τ-bench E2E (Session 4) on API key; the golden-set regression
#      below uses synthetic trials only, so it runs key-free.
# (kr) 선택 항목인 라이브 τ-bench E2E(Session 4)만 API 키 여부로 제어합니다.
#      아래 골든셋 회귀는 합성 trial만 사용하므로 키 없이 실행됩니다.
HAS_API = bool(os.environ.get("EXAONE_API_KEY", "").strip())
ROOT = exaone.project_root()
TRACK08 = ROOT / "recipes" / "track08_evaluation_m1_m10"
DATA = TRACK08 / "data"
out_dir = TRACK08 / "_out"
out_dir.mkdir(parents=True, exist_ok=True)

from eval.metrics.types import ToolCallRecord, TrialResult


def trial(task_id: str, **kw):
    defaults = dict(
        trial_id=f"t-{task_id}",
        task_id=task_id,
        dataset="track08.golden",
        runner="harness",
        final_content="",
        tool_calls=[],
        turns=1,
        input_tokens=50,
        output_tokens=30,
    )
    defaults.update(kw)
    return TrialResult(**defaults)


# (en) Build trial kwargs from a golden-seed row (dict answers -> JSON for M6).
# (kr) 골든 시드 행으로 trial kwargs를 만듭니다(dict 정답 -> M6용 JSON).
def trial_fields_from_row(row: dict) -> dict:
    if row.get("trial_content"):
        return {"final_content": row["trial_content"]}
    ans = row.get("expected_answer")
    if isinstance(ans, dict):
        return {
            "final_structured": ans,
            "final_content": json.dumps(ans, ensure_ascii=False),
        }
    if ans is not None:
        return {"final_content": str(ans)}
    return {"final_content": ""}


print("exaone", exaone.__version__, "| HAS_API =", HAS_API)
print("out_dir", out_dir.relative_to(ROOT))

**출력 해석:** 설정 셀이 노트북 실행 조건과 산출물 위치를 고정합니다.

- `exaone <버전> | HAS_API = True/False` — `eval.metrics` import가 성공했다는 뜻이며, Session 1–2·5 회귀는 **API 키 없이도** 실행됩니다.
- `out_dir recipes/track08_evaluation_m1_m10/_out` — Jupyter 실행 위치와 무관하게 Track 08 폴더 아래 산출물을 저장합니다.
- `HAS_API = True`이면 Session 4의 실행 조건 중 하나를 만족합니다. 실제 실행 여부는 Session 4의 `RUN_TAUBENCH_E2E` 플래그가 결정합니다.

## Session 1. 골든 시드 → 합성 trial → M1/M5/M6/M9/M3 회귀

**테스트 시나리오** — `golden_seed.jsonl`의 `seed_rows` 22건을 사용합니다. 카테고리별로 5개 지표를 **오프라인**에서 계산하며, 모든 행이 적어도 하나의 지표에 쓰이도록 구성했습니다.

| category | 검증 지표 | 무엇을 |
|---|---|---|
| `exact` | M1 | 정답 일치 |
| `schema` | M6 | 필수 키 (strict/loose) |
| `faithfulness` | M9 | 근거 충실도(교육용 스텁 judge) |
| `abstain` | M5 | 도구를 호출하지 않아야 하는 상황의 절제 |
| `tool` | M3 | 옳은 도구 선택 |

각 행에는 `category`가 하나만 있지만, 실제 채점 지표는 `expected_answer`·`required_keys`·`grounding_context`·`expected_no_tools`·`gold_tools` 같은 **라벨 필드**로 결정됩니다. 그래서 `schema` 4건뿐 아니라, `exact` 중 g01·g02도 `required_keys`를 가지고 있어 M6 분모에 함께 들어갑니다.

> **⚠️ 개념 구분 — 골든셋(라벨) vs 검사 대상(에이전트) 출력:** 골든셋의 한 행은 *입력 + gold 라벨*(query·정답·필수 키·근거·기대 도구)입니다. 회귀 점수는 **검사 대상 에이전트의 출력**을 이 라벨과 비교해 매깁니다. 실전에서는 `내 에이전트`를 골든 입력으로 실행한 뒤 *그 출력*을 채점합니다(end-to-end 회귀는 **Track 10 캡스톤**에서 다룹니다). 이 교육용 노트북에서는 이해를 돕기 위해 trial(출력)을 시드 안에 함께 둡니다. 22건 중 **11건은 라벨에서 파생되어 통과하도록 구성**했고, **11건은 `trial_content`로 직접 작성한, 실패할 수 있는 출력**입니다(g13·g15·g22처럼 점수에 영향을 주는 행). 그래서 M1 0.857, M6 0.833처럼 1.0이 아닌 값이 나옵니다.

> **좋은 골든셋 행의 조건:** ① 팀이 합의한 *고정* 정답 ② 카테고리별 대표 사례(정답·절제·도구·스키마·근거) ③ 통과/실패 판정이 흔들리지 않는 라벨 ④ 회귀를 감지하기 위한 **의도적 실패 사례**(예: g22의 잘못된 도구 호출, g13의 근접 오답)도 포함합니다.

각 시드 행의 정답, 필수 키, 근거 컨텍스트, 기대 도구를 읽어 해당 지표를 계산합니다.

### Session 1-1. seed_rows 로드 · 회귀 채점

**하는 일:** `seed_rows`를 읽고 카테고리별로 M1/M5/M6/M9/M3 회귀 점수를 계산합니다.

**입력:** `data/golden_seed.jsonl` (22건)

**정상:** `시나리오:` + 카테고리별 건수 + `지표별 채점 대상 수` + `rows 22 | scored 22/22 | M1 mean 0.857 | M5 1.0 | M3 0.667`

**의미:** 팀 mini-check 기준(M1·M6·M5 평균)에 들어갈 입력 데이터이며, 모든 행이 실제로 채점됩니다.

In [ ]:
from eval.metrics import (
    m1_task_success,
    m6_schema_adherence,
    m5_abstention,
    m3_tool_selection,
)
from eval.metrics.m1_task_success import TaskGold
from eval.metrics.m6_schema_adherence import SchemaSpec
from eval.metrics.m9_faithfulness import LengthRatioJudge

METRIC_ORDER = ["M1", "M6", "M9", "M5", "M3"]


def metric_names_for_row(row: dict) -> list[str]:
    names = []
    if row.get("expected_answer") is not None:
        names.append("M1")
    if row.get("required_keys"):
        names.append("M6")
    if row.get("grounding_context"):
        names.append("M9")
    if row.get("expected_no_tools"):
        names.append("M5")
    if row.get("gold_tools"):
        names.append("M3")
    return names


seed_rows = [
    json.loads(line)
    for line in (DATA / "golden_seed.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]
assert len(seed_rows) >= 20, len(seed_rows)
from collections import Counter

cats = Counter(r.get("category", "?") for r in seed_rows)
metric_coverage = Counter(
    metric for row in seed_rows for metric in metric_names_for_row(row)
)
unscored_seed_ids = [row["id"] for row in seed_rows if not metric_names_for_row(row)]
print("시나리오: 골든 시드 회귀", len(seed_rows), "건 (golden_seed.jsonl)")
for cat, n in sorted(cats.items()):
    print(f"  {cat}: {n}건")
print("지표별 채점 대상 수:", {name: metric_coverage[name] for name in METRIC_ORDER})
print("채점 예시:")
for row in seed_rows[:5]:
    print(
        f"  {row['id']} ({row.get('category')}): {', '.join(metric_names_for_row(row))}"
    )
assert not unscored_seed_ids, unscored_seed_ids

results = []
m1_scores, m6_scores, m9_scores = [], [], []
# (en) Collect abstain/tool trials so M5 (abstention) and M3 (tool selection) are scored too —
#      every category contributes to a metric (no dead padding).
# (kr) abstain/tool trial도 모아 M5(절제)·M3(도구 선택)까지 채점합니다.
#      모든 카테고리가 적어도 하나의 지표에 쓰이도록 합니다.
abstain_trials, abstain_expected = [], {}
tool_trials, tool_gold = [], {}
scored_ids = set()

for row in seed_rows:
    tid = row["id"]
    t_calls = []
    if row.get("wrong_tool"):
        t_calls = [ToolCallRecord(name=row["wrong_tool"], arguments={})]
    elif row.get("gold_tools"):
        t_calls = [
            ToolCallRecord(name=g["name"], arguments=g["arguments"])
            for g in row["gold_tools"]
        ]
    tr = trial(tid, **trial_fields_from_row(row), tool_calls=t_calls)

    rec = {
        "id": tid,
        "category": row.get("category"),
        "M1": None,
        "M6": None,
        "M9": None,
        "M5": None,
        "M3": None,
        "metrics": metric_names_for_row(row),
    }
    if row.get("expected_answer") is not None:
        rec["M1"] = m1_task_success.score_trial_exact(
            tr, TaskGold(task_id=tid, answer=row["expected_answer"])
        )
        m1_scores.append(rec["M1"])
        scored_ids.add(tid)
    if row.get("required_keys"):
        s, l = m6_schema_adherence.score_trial(
            tr, SchemaSpec(required_keys=row["required_keys"])
        )
        rec["M6"] = {"strict": s, "loose": l}
        m6_scores.append(1.0 if l else 0.0)
        scored_ids.add(tid)
    if row.get("grounding_context"):
        rec["M9"] = LengthRatioJudge()(
            trial=tr, gold={"context": row["grounding_context"]}
        )
        m9_scores.append(rec["M9"])
        scored_ids.add(tid)
    if row.get("expected_no_tools"):
        abstain_trials.append(tr)
        abstain_expected[tid] = True
        scored_ids.add(tid)
    if row.get("gold_tools"):
        tool_trials.append(tr)
        tool_gold[tid] = [
            ToolCallRecord(name=g["name"], arguments=g["arguments"])
            for g in row["gold_tools"]
        ]
        scored_ids.add(tid)
    results.append(rec)

# (en) M5/M3 are batch metrics over the abstain/tool subsets.
# (kr) M5/M3은 abstain/tool 부분집합에 대한 배치 지표입니다.
m5 = m5_abstention.compute(abstain_trials, abstain_expected)
m3 = m3_tool_selection.compute(tool_trials, tool_gold)
m1_failed_ids = [r["id"] for r in results if r["M1"] == 0.0]
m6_failed_ids = [
    r["id"] for r in results if isinstance(r["M6"], dict) and not r["M6"]["loose"]
]
tool_mismatch_ids = [row["id"] for row in seed_rows if row.get("wrong_tool")]
print(
    "rows",
    len(results),
    "| scored",
    len(scored_ids),
    "/",
    len(seed_rows),
    "| M1 mean",
    round(sum(m1_scores) / max(len(m1_scores), 1), 3),
    "| M5",
    round(m5.value, 3),
    "| M3",
    round(m3.value, 3),
)
print(
    "의도적 실패 사례:",
    {"M1": m1_failed_ids, "M6": m6_failed_ids, "M3": tool_mismatch_ids},
)

**출력 해석:** 22건이 카테고리별로 5개 지표에 채점됩니다.

- 카테고리 분포는 `exact 7 · schema 4 · faithfulness 5 · abstain 3 · tool 3`입니다. **모든 행이 M1/M6/M9/M5/M3 중 하나에 쓰입니다.**
- `지표별 채점 대상 수`는 라벨 필드 기준 분모입니다. `M6=6`처럼 카테고리 수와 지표 분모가 꼭 같지는 않습니다.
- `채점 예시`는 처음 5개 행이 어떤 지표에 들어가는지 보여줍니다. g01·g02는 exact 행이지만 schema 라벨도 있어 M1과 M6에 동시에 기여합니다.
- `M1 mean 0.857` — `expected_answer`가 있는 7건 중 6건이 일치합니다. g13은 정답 "환불요청" 대신 "환불 관련 문의로 보입니다."를 출력하므로 **exact 불일치**입니다. exact는 의미 유사도가 아니라 정규화된 값의 일치를 봅니다.
- `의도적 실패 사례`는 회귀 점수에 영향을 주는 행입니다. g13(M1)·g15(M6)·g22(M3)가 있어 모든 점수가 1.0으로 끝나지 않습니다.
- `M5 1.0`은 abstain 3건 모두 도구를 호출하지 않았다는 뜻이고, `M3 0.667`은 tool 3건 중 g22가 weather를 잘못 호출했기 때문입니다.

## Session 2. 회귀 요약 + mini-check

팀 기준값(여기서는 M1·M6·M5 평균 ≥ 0.8 + 모든 행 채점)을 정해, 점수가 떨어졌을 때 배포를 멈출 근거를 마련합니다.


### Session 2-1. 회귀 요약 + mini-check

**하는 일:** 지표별 평균과 분모를 요약하고, 회귀 게이트(mini-check)를 계산합니다.

**정상:** `summary` JSON + `all_rows_scored`/`M1`/`M6`/`M5`/`M9_present` … `PASS` 5줄

**의미:** 팀 기준값(M1·M6·M5 평균 ≥ 0.8 + 모든 행 채점)을 명시해, 점수가 떨어졌을 때 배포를 멈출 근거를 확보합니다.

In [ ]:
gate_thresholds = {"M1_mean": 0.8, "M6_loose_mean": 0.8, "M5_mean": 0.8, "M9_min_n": 3}

summary = {
    "n_cases": len(results),
    "n_scored": len(scored_ids),
    "metric_coverage": {name: metric_coverage[name] for name in METRIC_ORDER},
    "M1_mean": sum(m1_scores) / max(len(m1_scores), 1),
    "M6_loose_mean": sum(m6_scores) / max(len(m6_scores), 1),
    "M9_mean": sum(m9_scores) / max(len(m9_scores), 1),
    "M5_mean": m5.value,
    "M3_mean": m3.value,
    "denominators": {
        "M1": len(m1_scores),
        "M6": len(m6_scores),
        "M9": len(m9_scores),
        "M5": m5.n,
        "M3": m3.n,
    },
    "gate_thresholds": gate_thresholds,
    "sample_ids": [r["id"] for r in results[:5]],
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

# (en) Gate on EVERY row being scored (no dead padding) + per-metric thresholds. M9 is
#      count-gated (not value-gated) because its judge here is a test-only stub.
# (kr) 모든 행이 채점되는지와 지표별 기준값을 함께 검사합니다. M9 judge는
#      테스트용 스텁이므로 점수가 아니라 채점 건수만 확인합니다.
checks = [
    ("all_rows_scored", summary["n_scored"] == summary["n_cases"]),
    ("M1_mean>=0.8", summary["M1_mean"] >= gate_thresholds["M1_mean"]),
    ("M6_mean>=0.8", summary["M6_loose_mean"] >= gate_thresholds["M6_loose_mean"]),
    ("M5_mean>=0.8", summary["M5_mean"] >= gate_thresholds["M5_mean"]),
    ("M9_present>=3", len(m9_scores) >= gate_thresholds["M9_min_n"]),
]
for name, ok in checks:
    print(name, "PASS" if ok else "FAIL")
print("M3 (informational; g22 is a deliberate wrong-tool case):", round(m3.value, 3))
assert all(ok for _, ok in checks), "golden-set regression failed"

**출력 해석:** 회귀 게이트 5개 항목이 모두 PASS로 통과합니다.

- `all_rows_scored` — 22행이 모두 채점됐습니다. 단순히 `cases≥20`만 보는 게이트 대신, 실제로 평가에 쓰이지 않는 행이 없는지 확인합니다. `metric_coverage`와 `denominators`는 `M1=7·M6=6·M9=5·M5=3·M3=3` 분모를 보여줍니다.
- `M6=6`은 `schema` 4건뿐 아니라, `exact` 중 g01·g02도 `required_keys`를 가지고 있기 때문입니다.
- `gate_thresholds`는 팀이 정한 기준값입니다. 여기서는 M1·M6·M5 평균은 0.8 이상, M9는 최소 3건 채점으로 둡니다.
- `M1 0.857·M6 0.833·M5 1.0`은 모두 0.8 이상이라 통과합니다. g13(M1)과 g15(M6, "not json")의 실패 덕분에 단순히 1.0만 나오는 예제가 아니라는 점도 확인할 수 있습니다.
- `M9_present`는 **점수가 아니라 건수(≥3)** 만 검사합니다. 교육용 `LengthRatioJudge` 스텁의 0.533을 실제 품질 게이트로 쓰면 안 되기 때문입니다.
- `M3 0.667`은 g22의 의도적인 잘못된 도구 호출 때문에 참고용으로만 출력하며, 게이트에는 넣지 않습니다.

## Session 2b. 게이트가 작동하는 순간 — 회귀를 감지해 배포를 막기

mini-check가 "통과"만 보여주면 게이트의 핵심 역할, 즉 점수가 떨어졌을 때 **배포를 막는 동작**을 확인하기 어렵습니다. Session 2의 기준값을 `release_gate` 함수로 정의하고, **같은 골든셋**을 *회귀된 릴리스 후보*(성능이 나빠진 모델의 출력)로 다시 채점해 게이트가 배포를 막는 과정을 확인합니다.

In [ ]:
# (en) Reusable gate: pass only if every threshold holds (M9 is count-gated; M3 informational).
# (kr) 재사용 가능한 게이트입니다. 모든 기준값을 만족할 때만 통과합니다(M9는 건수, M3는 참고용).
def release_gate(m1_mean, m6_mean, m5_mean, m9_n):
    fails = []
    if m1_mean < gate_thresholds["M1_mean"]:
        fails.append(f"M1 {m1_mean:.2f} < {gate_thresholds['M1_mean']:.1f}")
    if m6_mean < gate_thresholds["M6_loose_mean"]:
        fails.append(f"M6 {m6_mean:.2f} < {gate_thresholds['M6_loose_mean']:.1f}")
    if m5_mean < gate_thresholds["M5_mean"]:
        fails.append(f"M5 {m5_mean:.2f} < {gate_thresholds['M5_mean']:.1f}")
    if m9_n < gate_thresholds["M9_min_n"]:
        fails.append(f"M9 채점 {m9_n}건 < {gate_thresholds['M9_min_n']}건")
    return (not fails, fails)


# (en) Regressed candidate: the worse model now fails 4 of 7 exact golden cases. We score
#      ITS OUTPUTS against the SAME gold labels (the real regression workflow: output vs gold).
# (kr) 회귀된 후보입니다. 성능이 나빠진 모델이 exact 골든 7건 중 4건을 틀렸다고 가정합니다.
#      이 출력을 같은 gold 라벨로 채점합니다(실제 회귀 흐름: 에이전트 출력 vs gold).
exact_rows = [r for r in seed_rows if r.get("category") == "exact"]
regressed = {
    row["id"]: ("처리 실패" if i < 4 else row["expected_answer"])
    for i, row in enumerate(exact_rows)
}
reg_m1 = []
for row in exact_rows:
    out = regressed[row["id"]]
    out_fields = (
        {"final_structured": out, "final_content": json.dumps(out, ensure_ascii=False)}
        if isinstance(out, dict)
        else {"final_content": str(out)}
    )
    reg_m1.append(
        m1_task_success.score_trial_exact(
            trial(row["id"], **out_fields),
            TaskGold(task_id=row["id"], answer=row["expected_answer"]),
        )
    )
reg_m1_mean = sum(reg_m1) / len(reg_m1)

# (en) Same gate, two candidates: the current release passes; the regressed one is blocked.
# (kr) 같은 게이트를 두 후보에 적용합니다. 현재 릴리스는 통과하고, 회귀된 후보는 차단됩니다.
base_ok, _ = release_gate(
    summary["M1_mean"], summary["M6_loose_mean"], summary["M5_mean"], len(m9_scores)
)
reg_ok, reg_fails = release_gate(
    reg_m1_mean, summary["M6_loose_mean"], summary["M5_mean"], len(m9_scores)
)
print(
    f"[현재 릴리스]  M1={summary['M1_mean']:.2f}  gate={'PASS ✅ 배포 허용' if base_ok else 'FAIL'}"
)
print(f"[회귀된 후보]  M1={reg_m1_mean:.2f}  gate={'PASS' if reg_ok else 'FAIL ❌'}")
if not reg_ok:
    print("🚫 배포 차단:", "; ".join(reg_fails))

**출력 해석:** 같은 게이트를 두 후보에 적용하면 하나는 통과하고 하나는 차단됩니다.

- `[현재 릴리스] M1=0.86 → PASS ✅ 배포 허용` — Session 2의 기준 결과가 기준값을 넘습니다.
- `[회귀된 후보] M1=0.43 → FAIL ❌` — 성능이 나빠진 모델이 exact 7건 중 4건을 틀려 M1이 0.8 아래로 떨어지고, `🚫 배포 차단: M1 0.43 < 0.8`이 출력됩니다.
- 핵심은 게이트가 *점수를 계산하는 것*에서 끝나지 않고, **점수가 떨어졌을 때 릴리스를 막는 역할**을 한다는 점입니다. 여기서도 회귀된 후보의 **출력을 같은 gold 라벨로** 채점합니다. 라벨끼리 비교하는 것이 아니라, 에이전트 출력과 gold 라벨을 비교하는 것이 실제 회귀 흐름입니다.

## Session 3. (선택) 실제 데이터셋 미리보기 — BFCL · IFEval · τ-bench

`eval.datasets.load_dataset`을 사용해 표준 벤치마크를 로드해 봅니다. 로컬 캐시나 선택 설치가 없으면 오류 없이 건너뜁니다.


### Session 3-1. 표준 데이터셋 미리보기

**하는 일:** BFCL·IFEval·τ-bench를 `load_dataset`으로 2건씩 미리 확인합니다.

**정상:** 각 데이터셋의 `task_id`/`query` 미리보기, 또는 캐시·설치가 없을 때 `{"skip": ...}`

**의미:** 합성 골든셋이 실제 표준 벤치마크와 어떻게 연결되는지 확인합니다. 로컬 캐시나 선택 설치가 없으면 오류 없이 건너뜁니다.

In [ ]:
from eval.datasets import load_dataset

dataset_preview = {}
for name in ("bfcl_v3.simple", "ifeval", "tau_bench.retail"):
    try:
        rows = load_dataset(name, limit=2)
        dataset_preview[name] = [
            {"task_id": r.task_id, "query": (r.query or "")[:60]} for r in rows
        ]
    except Exception as exc:
        dataset_preview[name] = {"skip": str(exc)[:120]}
print(json.dumps(dataset_preview, ensure_ascii=False, indent=2))

**출력 해석:** 표준 벤치마크 로드를 시도합니다.

- `bfcl_v3.simple`·`ifeval` — 로컬 캐시가 있으면 `task_id`/`query` 2건이 보입니다(실제 BFCL 함수 호출·IFEval 지시문).
- `tau_bench.retail` — 설치나 캐시가 없으면 `{"skip": "tau-bench is not installed …"}`처럼 표시되고, 예외는 위로 전파되지 않습니다.
- 합성 골든셋과 달리 이 데이터셋들은 외부 의존성이 있으므로 환경에 따라 일부만 로드될 수 있습니다.

## Session 4. (선택) `eval.run` τ-bench E2E

τ-bench retail은 **멀티턴 라이브 벤치마크**입니다. 에이전트와 *사용자 시뮬레이터(또 다른 LLM)*가 여러 차례 대화하면서 retail 도구를 호출하므로, 태스크 하나에도 **수십~수백 번의 순차 LLM 호출**이 발생해 **수 분**이 걸립니다. 이 노트북의 핵심인 골든셋 회귀는 바로 끝나야 하므로, 느린 외부 벤치마크는 **기본 비활성화**(`RUN_TAUBENCH_E2E=False`) 상태로 두어 노트북이 몇 초 안에 통과하도록 했습니다. "골든셋·지표가 실제 표준 벤치마크와 연결된다"는 점은 Session 3 미리보기에서 이미 빠르게 확인했습니다. 이 섹션에서는 실제 실행 명령을 문서화하고, 학습자가 선택한 경우에만 실행합니다. 실제 에이전트 E2E 회귀는 **Track 10 캡스톤**에서 다룹니다. tau-bench는 `uv pip install 'exaone-cookbook[eval-taubench]'`로 설치합니다.

### Session 4-1. (선택) τ-bench E2E 실행

**하는 일:** 기본 상태에서는 `eval.run` 명령과 실행하지 않는 이유만 출력하고 바로 끝납니다. `RUN_TAUBENCH_E2E=True`, API 키, tau-bench 설치가 모두 준비되어 있으면 최소 설정(`--limit 1 --pass-k-trials 1`)으로 실제 실행하고, 경과 시간을 주기적으로 출력해 진행 상황을 보여줍니다.

**정상:** 기본 → `[NOT RUN] τ-bench E2E (opt-in) — RUN_TAUBENCH_E2E=False` + 실행 명령. 플래그를 켜면 → `… 실행 중 (Ns 경과)`가 주기적으로 출력된 뒤 `Saved: …/eval/reports/taubench/*`가 표시됩니다(설치된 경우). 설치되어 있지 않으면 `ImportError …`를 잡아 오류 상태로 남깁니다.

**의미:** 골든셋 회귀를 실제 τ-bench E2E와 연결하는 단계입니다. 다만 멀티턴이라 느리므로 **기본 비활성화** 상태로 두고, 실제 실행 여부는 학습자가 선택합니다. 어느 경우든 오프라인 회귀 산출물에는 영향이 없습니다.

In [ ]:
import subprocess
import time

# (en) τ-bench E2E is a multi-turn LIVE benchmark (agent + user-simulator loop) -> minutes,
#      and it is NOT the golden-set lesson. Off by default so the notebook passes in seconds;
#      flip the flag to actually run it (needs a key + the optional tau-bench install).
# (kr) τ-bench E2E는 멀티턴 라이브 벤치마크(에이전트+사용자 시뮬레이터 루프)라 수 분이 걸립니다.
#      이 노트북의 골든셋 학습과는 별도이므로 기본 비활성화 상태로 둡니다.
#      실제로 실행하려면 플래그를 켜야 합니다(키 + 선택적 tau-bench 설치 필요).
RUN_TAUBENCH_E2E = False  # ← True로 바꾸면 실제 라이브 E2E 실행(멀티턴이라 수 분 소요)

cmd = [
    sys.executable, "-m", "eval.run",
    "--dataset", "tau_bench.retail",
    "--limit", "1", "--pass-k-trials", "1",
    "--runners", "harness",
    "--out-dir", "eval/reports/taubench",
]

if not (RUN_TAUBENCH_E2E and HAS_API):
    # (en) Default: show the command + why it is gated; do NOT run the slow benchmark.
    # (kr) 기본 상태에서는 명령과 실행하지 않는 이유만 보여주고, 느린 벤치마크는 실행하지 않습니다.
    reason = "RUN_TAUBENCH_E2E=False" if not RUN_TAUBENCH_E2E else "no EXAONE_API_KEY"
    tau_run = {"status": "not_run", "reason": reason, "command": " ".join(cmd)}
    print("[NOT RUN] τ-bench E2E (opt-in) —", reason)
    print("  실행하려면: RUN_TAUBENCH_E2E=True (키 + tau-bench 설치 필요) — 멀티턴이라 수 분 소요")
    print("  명령:", " ".join(cmd))
else:
    # (en) eval.run is quiet mid-run, so we poll and print elapsed time as a heartbeat rather
    #      than a frozen-looking cell; the report table + Saved lines arrive at the end.
    # (kr) eval.run은 실행 중 로그가 적으므로, 멈춘 것처럼 보이지 않도록 경과 시간을 주기적으로 출력합니다.
    #      리포트 표와 Saved 줄은 마지막에 한꺼번에 출력됩니다.
    print("[RUN] τ-bench E2E — 멀티턴이라 수 분 걸립니다(경과 시간으로 진행 표시):")
    print(" ", " ".join(cmd))
    t0 = time.monotonic()
    proc = subprocess.Popen(
        cmd, cwd=ROOT, env={**os.environ, "PYTHONUNBUFFERED": "1"},
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    while proc.poll() is None:
        time.sleep(20)
        print(f"  … 실행 중 ({int(time.monotonic() - t0)}s 경과)")
    out = proc.stdout.read() or ""
    saved = [ln.replace(str(ROOT), "<repo-root>") for ln in out.splitlines() if ln.startswith("Saved:")]
    tau_run = {
        "returncode": proc.returncode,
        "status": "ok" if proc.returncode == 0 else "error",
        "report_paths": saved[-2:],
        "elapsed_s": round(time.monotonic() - t0, 1),
    }
    print(out[-600:])
    print("→ status:", tau_run["status"], f"| {tau_run['elapsed_s']}s | reports:", tau_run["report_paths"])

**출력 해석:** Session 4는 **기본 비활성화(즉시 종료)** 상태이며, 플래그를 켜야 실제로 실행됩니다.

- **기본(`RUN_TAUBENCH_E2E=False`)** → `[NOT RUN] … (opt-in)` + 실행 명령, `tau_run.status="not_run"`이 남습니다. 노트북은 몇 초 안에 통과합니다.
- **플래그를 켜고(True) 키가 있으면** → `… 실행 중 (Ns 경과)`가 주기적으로 출력됩니다(멀티턴이라 수 분 소요). 완료되면 `Saved: …`와 pass^k 리포트가 나오고, `status="ok"`, `elapsed_s`가 기록됩니다. 1개 태스크 스모크 테스트라 점수는 낮거나 0일 수 있으며, τ-bench retail 난이도를 고려하면 정상입니다. 핵심은 *게이트를 실제 벤치마크에 연결할 수 있다*는 점입니다.
- 플래그를 켰지만 tau-bench가 설치되어 있지 않으면 `ImportError`를 잡아 `status="error"`로 남깁니다. 어느 경우든 오프라인 회귀 산출물에는 영향이 없습니다.

## Session 5. 산출물 — `my_golden.jsonl` + `regression_report.json`


### Session 5-1. 산출물 저장 — my_golden.jsonl(골든셋) + regression_report.json(점수)

**하는 일:** 재사용할 골든셋(원본 시드 라벨)과 회귀 리포트(요약+행별 점수)를 Track 08 `_out/`에 따로 저장합니다.

**정상:** `saved: …/_out/my_golden.jsonl (golden set: 22 rows)`, `saved: …/_out/regression_report.json`

**의미:** 골든셋(라벨)과 점수 결과를 분리해, 다음 릴리스 때 **같은 골든셋**으로 회귀 검사를 반복할 수 있게 합니다.

In [ ]:
# (en) my_golden.jsonl IS the reusable golden set (original seed rows WITH gold labels).
# (kr) my_golden.jsonl은 재사용할 골든셋입니다(정답 라벨이 들어 있는 원본 시드 행).
jsonl_path = out_dir / "my_golden.jsonl"
with jsonl_path.open("w", encoding="utf-8") as fh:
    for row in seed_rows:
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")

# (en) The regression report holds the summary + per-row SCORES (not the golden labels).
# (kr) 회귀 리포트에는 요약과 행별 점수를 담습니다(정답 라벨이 아닙니다).
regression = {
    "track": "track08",
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "n_cases": len(results),
    "n_scored": len(scored_ids),
    "metric_coverage": summary["metric_coverage"],
    "gate_thresholds": gate_thresholds,
    "summary": {
        k: summary[k]
        for k in (
            "M1_mean",
            "M6_loose_mean",
            "M9_mean",
            "M5_mean",
            "M3_mean",
            "denominators",
        )
    },
    "dataset_preview_keys": list(dataset_preview.keys()),
    "tau_run": tau_run,
    "intentional_failures": {
        "M1": m1_failed_ids,
        "M6": m6_failed_ids,
        "M3": tool_mismatch_ids,
    },
    "scored_cases": results,
    "notes": {
        "M9_judge": "LengthRatioJudge is a TEST-ONLY token-overlap stub; M9 is a proxy, not a production faithfulness score.",
        "release_gate": "Track 10 capstone: run after each release; fail if M1/M6/M5 drop below team thresholds (M9 is count-gated only).",
    },
}
(out_dir / "regression_report.json").write_text(
    json.dumps(regression, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("saved:", jsonl_path.resolve(), "(golden set:", len(seed_rows), "rows)")
print("saved:", (out_dir / "regression_report.json").resolve())

**출력 해석:** 두 산출물이 서로 다른 목적에 맞게 저장됩니다.

- `my_golden.jsonl` — **재사용할 골든셋**입니다. query, 정답, 필수 키, 근거 컨텍스트, 기대 도구가 들어 있는 원본 22행을 보관합니다.
- `regression_report.json` — 이번 실행의 **요약과 행별 점수**입니다. M1–M3 평균·분모, `metric_coverage`, `gate_thresholds`, `intentional_failures`, `scored_cases`, `tau_run`, M9 스텁 주의 사항을 담습니다.
- `_out/`은 Track 08 폴더 아래에 고정되며 gitignore 대상이라 커밋되지 않습니다.

## Wrap-up. 마무리

이 노트북에서는 `golden_seed.jsonl`(22건)을 카테고리별 합성 trial로 만들고, **M1/M5/M6/M9/M3를 오프라인에서 채점**했습니다. 그런 다음 모든 행이 채점되는지와 기준값을 함께 확인하는 mini-check 회귀 게이트를 적용하고, **게이트가 회귀된 후보를 감지해 배포를 막는 장면(Session 2b)**까지 확인했습니다. 마지막으로 골든셋과 회귀 리포트를 따로 저장했습니다.

**핵심 정리**

- **골든셋은 회귀 게이트의 기준입니다.** `cases≥20` 같은 단순 건수가 아니라 *모든 행이 실제로 채점되는지*를 봐야 형식적인 행 수에 속지 않습니다.
- **분모는 라벨 필드가 정합니다.** `category`는 설명용 축이고, 실제 M1/M6/M9/M5/M3 분모는 `expected_answer`·`required_keys` 같은 gold 라벨 필드에서 나옵니다.
- **게이트의 역할은 점수 하락을 막는 것입니다.** Session 2b에서 회귀된 후보(M1 0.43)가 `🚫 배포 차단` 되는 장면이 게이트의 존재 이유를 보여줍니다.
- **지표마다 게이트 방식이 다릅니다.** M1·M6·M5는 값(≥0.8)으로 검사하고, M9는 judge가 스텁이므로 **건수(≥3)** 만 확인합니다. 스텁 점수를 릴리스 게이트에 직접 쓰면 안 됩니다.
- **산출물은 분리해서 저장합니다.** `my_golden.jsonl`(라벨)과 `regression_report.json`(점수·기준값·실패 사례)을 나누어 두면, 다음 릴리스 때 같은 골든셋을 그대로 재사용할 수 있습니다.

**한계 / 범위**

- 이 노트북의 trial은 **데모용 출력**입니다. 일부는 라벨에서 파생했고, 일부는 직접 작성했습니다. **실제 에이전트를 골든 입력에 실행하고 그 출력을 채점하는 end-to-end 회귀는 Track 10 캡스톤**에서 릴리스 게이트로 묶습니다. 여기서는 *채점과 게이트 메커니즘*에 집중합니다.
- M9 judge는 교육용 `LengthRatioJudge`(토큰 겹침) 스텁입니다. 따라서 `M9_mean 0.533`은 실제 충실도 점수가 아닙니다. 운영에서는 ExaoneAPIClient/NLI judge로 교체해야 합니다.
- Session 4 τ-bench E2E는 멀티턴 라이브 벤치마크라 수 분이 걸리므로 **기본 비활성화** 상태입니다. `RUN_TAUBENCH_E2E=True`, API 키, tau-bench 설치가 모두 준비된 경우에만 실행됩니다. 실제 에이전트 회귀는 Track 10에서 다룹니다.

## 체크포인트

- [ ] Session 2 회귀 mini-check 5개가 PASS인지 확인(모든 행 채점, M1·M6·M5 평균≥0.8, M9 건수≥3) — **키 없이 실행 가능**.
- [ ] Session 2b에서 회귀된 후보가 게이트 FAIL이 되고 `🚫 배포 차단`으로 막히는지 확인.
- [ ] `my_golden.jsonl`에는 골든셋 22행(라벨)이, `regression_report.json`에는 요약과 행별 점수가 기록되는지 확인.
- [ ] (선택) `RUN_TAUBENCH_E2E=True` + 키 + tau-bench 설치 시 Session 4가 `eval/reports/taubench`에 리포트를 남기는지 확인(기본은 `[NOT RUN]`).

**다음:** Track 10 — AX Capstones. 이 골든셋 회귀를 **실제 에이전트** 기준 릴리스 게이트로 연결해, 지표가 팀 기준값 아래로 떨어지면 배포를 멈추는 흐름을 다룹니다.